# H3 Hypothesis Testing: Age Groups

This notebook evaluates whether **reported median emergency department length of stay differs across age groups**.

- **Dataset:** `Age_Sex` aggregate data
- **Hypothesis test:** Weighted Kruskal-Wallis H test (non-parametric, multi-group)
- **Null hypothesis (H₀):** There is no significant difference in reported median ED LOS among age groups.
- **Alternative hypothesis (H₁):** There is a significant difference in reported median ED LOS among age groups.

> **Aggregate-data interpretation:** The dataset contains grouped/aggregate observations. Results describe differences in reported aggregate median LOS across age-group categories and do **not** imply patient-level differences or individual patient behavior.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# Locate the repository root and Age_Sex dataset

possible_roots = [Path.cwd()] + list(Path.cwd().parents[:5])

repo_root = None

for candidate in possible_roots:
    if (candidate / "data" / "cleaned dataset" / "Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx").exists():
        repo_root = candidate
        break

if repo_root is None:
    repo_root = Path(
        r"d:\term 5 capstone\capstone offical github repo\Capstone_Project-DAMO-6994-"
    )

WORKBOOK_PATH = (
    repo_root
    / "data"
    / "cleaned dataset"
    / "Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx"
)

CSV_PATH = (
    repo_root
    / "data"
    / "Explorer Dataset"
    / "Age_Sex.csv"
)

print("Resolved root:", repo_root)
print("Workbook path:", WORKBOOK_PATH)
print("CSV path:", CSV_PATH)
print("Workbook exists:", WORKBOOK_PATH.exists())
print("CSV exists:", CSV_PATH.exists())


Resolved root: d:\term 5 capstone\capstone offical github repo\Capstone_Project-DAMO-6994-
Workbook path: d:\term 5 capstone\capstone offical github repo\Capstone_Project-DAMO-6994-\data\cleaned dataset\Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx
CSV path: d:\term 5 capstone\capstone offical github repo\Capstone_Project-DAMO-6994-\data\Explorer Dataset\Age_Sex.csv
Workbook exists: True
CSV exists: False


In [4]:
if WORKBOOK_PATH.exists():
    df = pd.read_excel(
        WORKBOOK_PATH,
        sheet_name="Age_Sex"
    )
    print("Age_Sex data loaded from cleaned workbook.")

elif CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    print("Age_Sex data loaded from Explorer Dataset CSV.")

else:
    raise FileNotFoundError(
        "Age_Sex dataset not found. "
        "Ensure the cleaned workbook or Age_Sex.csv is available."
    )

print("Dataset loaded successfully.")


Age_Sex data loaded from cleaned workbook.
Dataset loaded successfully.


In [5]:
# Standardize column names where necessary

rename_map = {
    "median_length_of_stay_min": "median_los_minutes",
    "length_of_stay_hours": "median_los_hours"
}

df = df.rename(columns=rename_map)

required_columns = [
    "fiscal_year",
    "fiscal_year_start",
    "age_group",
    "ed_visits",
    "median_los_minutes"
]

missing_required = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("Required H3 columns are available.")
print("Columns:", df.columns.tolist())


Required H3 columns are available.
Columns: ['fiscal_year', 'fiscal_year_start', 'sex', 'age_group', 'population_category', 'ed_visits', 'median_los_minutes', 'median_los_hours']


In [6]:
print("FIRST 10 ROWS")
print("-" * 80)

display(df.head(10))


FIRST 10 ROWS
--------------------------------------------------------------------------------


,fiscal_year,fiscal_year_start,sex,age_group,population_category,ed_visits,median_los_minutes,median_los_hours
0,2003-2004,2003,Female,0–17,Pediatric Population,611154,101,1.6800
1,2003-2004,2003,Female,18–34,Young Adult Population,532768,127,2.1200
2,2003-2004,2003,Female,35–49,Adult Population,503478,129,2.1500
3,2003-2004,2003,Female,50–64,Pre-Senior Population,349504,141,2.3500
4,2003-2004,2003,Female,65–85+,Geriatric Population,478789,203,3.3800
5,2003-2004,2003,Male,0–17,Pediatric Population,702232,101,1.6800
6,2003-2004,2003,Male,18–34,Young Adult Population,471173,114,1.9000
7,2003-2004,2003,Male,35–49,Adult Population,509323,123,2.0500
8,2003-2004,2003,Male,50–64,Pre-Senior Population,351182,140,2.3300
9,2003-2004,2003,Male,65–85+,Geriatric Population,396791,188,3.1300


In [7]:
print("LAST 10 ROWS")
print("-" * 80)

display(df.tail(10))


LAST 10 ROWS
--------------------------------------------------------------------------------


,fiscal_year,fiscal_year_start,sex,age_group,population_category,ed_visits,median_los_minutes,median_los_hours
180,2021-2022,2021,Female,0–17,Pediatric Population,1293819,172,2.8700
181,2021-2022,2021,Female,18–34,Young Adult Population,1590665,204,3.4000
182,2021-2022,2021,Female,35–49,Adult Population,1301435,216,3.6000
183,2021-2022,2021,Female,50–64,Pre-Senior Population,1260136,226,3.7700
184,2021-2022,2021,Female,65–85+,Geriatric Population,1784078,326,5.4300
185,2021-2022,2021,Male,0–17,Pediatric Population,1345417,160,2.6700
186,2021-2022,2021,Male,18–34,Young Adult Population,1299236,186,3.1000
187,2021-2022,2021,Male,35–49,Adult Population,1202684,204,3.4000
188,2021-2022,2021,Male,50–64,Pre-Senior Population,1299479,229,3.8200
189,2021-2022,2021,Male,65–85+,Geriatric Population,1615080,307,5.1200


In [8]:
print("DATASET SHAPE")
print("-" * 80)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")


DATASET SHAPE
--------------------------------------------------------------------------------
Rows    : 190
Columns : 8


In [9]:
print("COLUMN NAMES")
print("-" * 80)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")


COLUMN NAMES
--------------------------------------------------------------------------------
01. fiscal_year
02. fiscal_year_start
03. sex
04. age_group
05. population_category
06. ed_visits
07. median_los_minutes
08. median_los_hours


In [10]:
print("DATA TYPES")
print("-" * 80)

display(
    pd.DataFrame({
        "Column": df.columns,
        "Data_Type": df.dtypes.astype(str).values
    })
)


DATA TYPES
--------------------------------------------------------------------------------


,Column,Data_Type
0,fiscal_year,object
1,fiscal_year_start,int64
2,sex,object
3,age_group,object
4,population_category,object
5,ed_visits,int64
6,median_los_minutes,int64
7,median_los_hours,float64


In [11]:
print("MISSING VALUES")
print("-" * 80)

missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": [
        df[column].isna().sum()
        for column in df.columns
    ],
    "Missing_Percentage": [
        df[column].isna().mean() * 100
        for column in df.columns
    ]
})

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Percentage"].round(2)
)

display(missing_summary)


MISSING VALUES
--------------------------------------------------------------------------------


,Column,Missing_Count,Missing_Percentage
0,fiscal_year,0,0.0000
1,fiscal_year_start,0,0.0000
2,sex,0,0.0000
3,age_group,0,0.0000
4,population_category,0,0.0000
5,ed_visits,0,0.0000
6,median_los_minutes,0,0.0000
7,median_los_hours,0,0.0000


In [12]:
print("DUPLICATE RECORDS")
print("-" * 80)

duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.2f}%"
)


DUPLICATE RECORDS
--------------------------------------------------------------------------------
Duplicate rows: 0
Duplicate percentage: 0.00%


In [13]:
print("NUMERICAL SUMMARY")
print("-" * 80)

display(
    df[
        ["ed_visits", "median_los_minutes"]
    ].describe()
)


NUMERICAL SUMMARY
--------------------------------------------------------------------------------


,ed_visits,median_los_minutes
count,190.0000,190.0000
mean,"925,068.1263",169.3737
std,"363,366.4171",47.1079
min,"349,504.0000",101.0000
25%,"574,678.5000",136.0000
50%,"929,175.0000",156.0000
75%,"1,207,598.0000",195.2500
max,"1,877,379.0000",326.0000


In [14]:
print("AGE GROUP CATEGORIES")
print("-" * 80)

age_group_counts = (
    df["age_group"]
    .value_counts(dropna=False)
)

display(age_group_counts.to_frame("Aggregate_Records"))


AGE GROUP CATEGORIES
--------------------------------------------------------------------------------


,Aggregate_Records
age_group,
0–17,38
18–34,38
35–49,38
50–64,38
65–85+,38


In [15]:
print("FISCAL YEAR COVERAGE")
print("-" * 80)

fiscal_years = (
    df["fiscal_year"]
    .dropna()
    .unique()
)

print("Fiscal years:")
print(sorted(fiscal_years))

print("\nRecords per fiscal year:")

display(
    df["fiscal_year"]
    .value_counts()
    .sort_index()
    .to_frame("Aggregate_Records")
)


FISCAL YEAR COVERAGE
--------------------------------------------------------------------------------
Fiscal years:
['2003-2004', '2004-2005', '2005-2006', '2006-2007', '2007-2008', '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022']

Records per fiscal year:


,Aggregate_Records
fiscal_year,
2003-2004,10
2004-2005,10
2005-2006,10
2006-2007,10
2007-2008,10
2008-2009,10
2009-2010,10
2010-2011,10
2011-2012,10


In [16]:
print("ED VISITS AND LOS SUMMARY")
print("-" * 80)

selected_columns = [
    "age_group",
    "ed_visits",
    "median_los_minutes"
]

if "median_los_hours" in df.columns:
    selected_columns.append("median_los_hours")

display(
    df[selected_columns]
    .describe(include="all")
    .T
)


ED VISITS AND LOS SUMMARY
--------------------------------------------------------------------------------


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age_group,190,5,0–17,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ed_visits,190.0000,NaN,NaN,NaN,"925,068.1263","363,366.4171","349,504.0000","574,678.5000","929,175.0000","1,207,598.0000","1,877,379.0000"
median_los_minutes,190.0000,NaN,NaN,NaN,169.3737,47.1079,101.0000,136.0000,156.0000,195.2500,326.0000
median_los_hours,190.0000,NaN,NaN,NaN,2.8226,0.7853,1.6800,2.2700,2.6000,3.2575,5.4300


In [ ]:
# H3 DATA VALIDATION

required_columns = [
    "age_group",
    "ed_visits",
    "median_los_minutes"
]

print("H3 DATA VALIDATION")
print("=" * 80)

for column in required_columns:
    print(
        f"{column:25} : "
        f"{'FOUND' if column in df.columns else 'MISSING'}"
    )

analysis_df = df[
    required_columns
].copy()

analysis_df = analysis_df.dropna(
    subset=required_columns
)

analysis_df = analysis_df[
    (analysis_df["ed_visits"] > 0) &
    (analysis_df["median_los_minutes"] >= 0)
].copy()

print("\nAfter validation:")
print(f"Records available : {len(analysis_df):,}")
print(f"Age groups        : {analysis_df['age_group'].nunique()}")
print(f"Total ED visits   : {analysis_df['ed_visits'].sum():,.0f}")
print(f"Missing values    : {analysis_df.isna().sum().sum()}")

print("\nAge groups:")

for group in analysis_df["age_group"].dropna().unique():
    print(f"  • {group}")


In [ ]:
# Keep only actual age-group categories.
# Any aggregate total row is excluded from the H3 comparison.

h3_df = df[
    ~df["age_group"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["total", "all ages", "all age groups"])
].copy()

h3_df = h3_df[
    [
        "fiscal_year",
        "fiscal_year_start",
        "age_group",
        "ed_visits",
        "median_los_minutes"
    ]
].copy()

h3_df = h3_df.dropna(
    subset=[
        "age_group",
        "ed_visits",
        "median_los_minutes"
    ]
)

h3_df = h3_df[
    (h3_df["ed_visits"] > 0) &
    (h3_df["median_los_minutes"] >= 0)
].copy()

print("=" * 80)
print("H3 ANALYSIS DATASET")
print("=" * 80)

print(f"Original records       : {len(df):,}")
print(f"Records after filtering: {len(h3_df):,}")
print(f"Records removed        : {len(df) - len(h3_df):,}")

print("\nAge groups included:")

for group in h3_df["age_group"].dropna().unique():
    print(
        f"  {group}: "
        f"{(h3_df['age_group'] == group).sum():,} records"
    )


In [ ]:
# Ordered age groups
# Use the source order when available; otherwise sort alphabetically.

observed_age_groups = list(
    h3_df["age_group"]
    .dropna()
    .astype(str)
    .unique()
)

known_age_order = [
    "0-4",
    "5-9",
    "10-14",
    "15-19",
    "20-24",
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60-64",
    "65-69",
    "70-74",
    "75-79",
    "80-84",
    "85+"
]

def age_group_sort_key(value):
    text = str(value).strip()

    match = re.match(r"^(\d+)", text)

    if match:
        return int(match.group(1))

    if text.lower() in {"unknown", "not stated", "not available"}:
        return 9999

    return 5000

ordered_age_groups = sorted(
    observed_age_groups,
    key=age_group_sort_key
)

print("Age-group order used for analysis:")
print(ordered_age_groups)


In [ ]:
print("WEIGHT VALIDATION")
print("=" * 80)

print("ED Visits data type:")
print(h3_df["ed_visits"].dtype)

print("\nNon-integer ED visit values:")

non_integer_weights = (
    h3_df["ed_visits"] % 1 != 0
).sum()

print(non_integer_weights)

print("\nMinimum ED visits:")
print(h3_df["ed_visits"].min())

print("\nMaximum ED visits:")
print(h3_df["ed_visits"].max())

print("\nTotal ED visits represented:")
print(
    f"{h3_df['ed_visits'].sum():,.0f}"
)


In [ ]:
def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    return np.average(
        values,
        weights=weights
    )


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative_weight = np.cumsum(weights)
    cutoff = weights.sum() / 2

    return values[
        np.searchsorted(
            cumulative_weight,
            cutoff,
            side="left"
        )
    ]


In [ ]:
h3_summary = (
    h3_df
    .groupby("age_group")
    .apply(
        lambda x: pd.Series({

            "records":
                len(x),

            "total_ed_visits":
                x["ed_visits"].sum(),

            "weighted_mean_los_minutes":
                weighted_mean(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "weighted_median_los_minutes":
                weighted_median(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "minimum_los_minutes":
                x["median_los_minutes"].min(),

            "maximum_los_minutes":
                x["median_los_minutes"].max()

        }),
        include_groups=False
    )
    .reset_index()
)

h3_summary["age_order"] = (
    h3_summary["age_group"]
    .astype(str)
    .map({
        group: i
        for i, group in enumerate(ordered_age_groups)
    })
)

h3_summary = (
    h3_summary
    .sort_values("age_order")
    .drop(columns="age_order")
)

display(h3_summary)


In [ ]:
# Weighted Kruskal-Wallis test for aggregate observations

def weighted_kruskal_wallis(
    data,
    group_col,
    value_col,
    weight_col
):

    test_data = data[
        [group_col, value_col, weight_col]
    ].dropna().copy()

    test_data = test_data[
        test_data[weight_col] > 0
    ].copy()

    if not np.all(
        np.isclose(
            test_data[weight_col] % 1,
            0
        )
    ):
        raise ValueError(
            "Weights must be integer visit counts."
        )

    test_data[weight_col] = (
        test_data[weight_col]
        .astype(np.int64)
    )

    N = int(
        test_data[weight_col].sum()
    )

    value_weights = (
        test_data
        .groupby(value_col)[weight_col]
        .sum()
        .sort_index()
    )

    cumulative_before = (
        value_weights
        .cumsum()
        .shift(fill_value=0)
    )

    weighted_ranks = (
        cumulative_before
        + (value_weights + 1) / 2
    )

    test_data["_rank"] = (
        test_data[value_col]
        .map(weighted_ranks)
    )

    group_stats = (
        test_data
        .groupby(group_col)
        .apply(
            lambda x: pd.Series({

                "weight":
                    x[weight_col].sum(),

                "rank_sum":
                    (
                        x["_rank"]
                        * x[weight_col]
                    ).sum()

            }),
            include_groups=False
        )
    )

    H_raw = (
        12 /
        (N * (N + 1))
    ) * (
        (
            group_stats["rank_sum"] ** 2
            /
            group_stats["weight"]
        ).sum()
    ) - (
        3 * (N + 1)
    )

    tie_weights = value_weights.values

    tie_correction = (
        1 -
        (
            np.sum(
                tie_weights ** 3 -
                tie_weights
            )
            /
            (
                N ** 3 -
                N
            )
        )
    )

    H_corrected = (
        H_raw /
        tie_correction
    )

    degrees_freedom = (
        group_stats.shape[0] - 1
    )

    p_value = stats.chi2.sf(
        H_corrected,
        degrees_freedom
    )

    return {
        "H": H_corrected,
        "df": degrees_freedom,
        "p_value": p_value,
        "N_weight": N,
        "tie_correction": tie_correction,
        "group_stats": group_stats
    }


In [ ]:
# ============================================================
# WEIGHTED KRUSKAL-WALLIS TEST
# ============================================================

h3_result = weighted_kruskal_wallis(
    data=h3_df,
    group_col="age_group",
    value_col="median_los_minutes",
    weight_col="ed_visits"
)

print("H3 - WEIGHTED KRUSKAL-WALLIS TEST")
print("-" * 80)

print(
    f"Test statistic (H): "
    f"{h3_result['H']:.6f}"
)

print(
    f"Degrees of freedom: "
    f"{h3_result['df']}"
)

print(
    f"P-value: "
    f"{h3_result['p_value']:.10f}"
)

print(
    f"Total frequency weight: "
    f"{h3_result['N_weight']:,}"
)

print(
    f"Tie correction: "
    f"{h3_result['tie_correction']:.10f}"
)


In [ ]:
ALPHA = 0.05

p_value = h3_result["p_value"]

print("=" * 80)
print("H3 HYPOTHESIS DECISION")
print("=" * 80)

print("Significance level (α):", ALPHA)
print(f"P-value: {p_value:.10f}")

if p_value < ALPHA:

    print("\nDecision: REJECT H₀")

    print("\nConclusion:")

    print(
        "There is statistically significant evidence "
        "that reported median ED length of stay differs "
        "across age groups in the aggregate Age_Sex data."
    )

else:

    print("\nDecision: FAIL TO REJECT H₀")

    print("\nConclusion:")

    print(
        "There is insufficient statistical evidence "
        "to conclude that reported median ED length "
        "of stay differs across age groups in the "
        "aggregate Age_Sex data."
    )


In [ ]:
H = h3_result["H"]

k = h3_result["df"] + 1

N = h3_result["N_weight"]

epsilon_squared = (
    H - k + 1
) / (
    N - k
)

print("H3 EFFECT SIZE")
print("-" * 80)

print(
    f"Epsilon squared (ε²): "
    f"{epsilon_squared:.10f}"
)


In [ ]:
h3_result_table = pd.DataFrame({

    "Hypothesis": [
        "H3"
    ],

    "Research_Question": [
        "Does reported median ED LOS differ across age groups?"
    ],

    "Test": [
        "Weighted Kruskal-Wallis"
    ],

    "H_Statistic": [
        H
    ],

    "Degrees_of_Freedom": [
        h3_result["df"]
    ],

    "P_Value": [
        p_value
    ],

    "Epsilon_Squared": [
        epsilon_squared
    ],

    "Alpha": [
        ALPHA
    ],

    "Decision": [
        "Reject H0"
        if p_value < ALPHA
        else "Fail to Reject H0"
    ]

})

display(h3_result_table)


In [ ]:
# ============================================================
# H3 VISUALIZATION DATA
# ============================================================

h3_df["age_group"] = pd.Categorical(
    h3_df["age_group"],
    categories=ordered_age_groups,
    ordered=True
)

h3_df = h3_df.sort_values("age_group")

chart_labels = [
    str(x)
    for x in ordered_age_groups
]

print("H3 visualization data prepared.")


In [ ]:
# NUMBER OF AGGREGATE RECORDS BY AGE GROUP

record_counts = (
    h3_df
    .groupby("age_group", observed=True)
    .size()
    .reindex(ordered_age_groups)
)

plt.figure(figsize=(12, 6))

bars = plt.bar(
    chart_labels,
    record_counts.values
)

plt.title(
    "Aggregate Records by Age Group"
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "Number of Aggregate Records"
)

plt.xticks(rotation=35)

for bar, value in zip(
    bars,
    record_counts.values
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.show()


In [ ]:
# TOTAL ED VISITS BY AGE GROUP

visit_summary = (
    h3_df
    .groupby("age_group", observed=True)["ed_visits"]
    .sum()
    .reindex(ordered_age_groups)
)

plt.figure(figsize=(12, 6))

bars = plt.bar(
    chart_labels,
    visit_summary.values
)

plt.title(
    "Total Reported ED Visits by Age Group"
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "Total ED Visits"
)

plt.xticks(rotation=35)

for bar, value in zip(
    bars,
    visit_summary.values
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,.0f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.show()


In [ ]:
# WEIGHTED MEDIAN LOS BY AGE GROUP

weighted_los = (
    h3_summary
    .set_index("age_group")
    .reindex(ordered_age_groups)
    ["weighted_median_los_minutes"]
)

plt.figure(figsize=(12, 6))

bars = plt.bar(
    chart_labels,
    weighted_los.values
)

plt.title(
    "Weighted Median Reported ED Length of Stay by Age Group"
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "Weighted Median LOS (Minutes)"
)

plt.xticks(rotation=35)

for bar, value in zip(
    bars,
    weighted_los.values
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.0f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.show()


In [ ]:
# WEIGHTED MEAN VS WEIGHTED MEDIAN LOS

mean_los = (
    h3_summary
    .set_index("age_group")
    .reindex(ordered_age_groups)
    ["weighted_mean_los_minutes"]
)

median_los = (
    h3_summary
    .set_index("age_group")
    .reindex(ordered_age_groups)
    ["weighted_median_los_minutes"]
)

x = np.arange(len(ordered_age_groups))

width = 0.36

plt.figure(figsize=(13, 6))

plt.bar(
    x - width / 2,
    mean_los.values,
    width,
    label="Weighted Mean"
)

plt.bar(
    x + width / 2,
    median_los.values,
    width,
    label="Weighted Median"
)

plt.xticks(
    x,
    chart_labels,
    rotation=35
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "LOS (Minutes)"
)

plt.title(
    "Weighted Mean and Median Reported ED LOS by Age Group"
)

plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# LOS DISTRIBUTION BY AGE GROUP

box_data = []

for group in ordered_age_groups:

    values = (
        h3_df.loc[
            h3_df["age_group"] == group,
            "median_los_minutes"
        ]
        .dropna()
        .values
    )

    box_data.append(values)

plt.figure(figsize=(13, 6))

plt.boxplot(
    box_data,
    labels=chart_labels,
    showmeans=True
)

plt.title(
    "Distribution of Reported Median ED LOS by Age Group"
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "Reported Median LOS (Minutes)"
)

plt.xticks(rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
# REPORTED ED LOS TREND ACROSS AGE GROUPS BY FISCAL YEAR

year_age = (
    h3_df
    .groupby(
        ["fiscal_year_start", "age_group"],
        observed=True
    )
    .apply(
        lambda x: pd.Series({
            "weighted_mean_los":
                weighted_mean(
                    x["median_los_minutes"],
                    x["ed_visits"]
                )
        }),
        include_groups=False
    )
    .reset_index()
)

plt.figure(figsize=(14, 7))

for group in ordered_age_groups:

    subset = year_age[
        year_age["age_group"] == group
    ]

    plt.plot(
        subset["fiscal_year_start"],
        subset["weighted_mean_los"],
        marker="o",
        label=str(group)
    )

plt.title(
    "Reported ED LOS Trend Across Age Groups by Fiscal Year"
)

plt.xlabel(
    "Fiscal Year Start"
)

plt.ylabel(
    "Weighted Mean Reported LOS (Minutes)"
)

plt.legend(
    title="Age Group",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ESTIMATED RESOURCE BURDEN INDEX BY AGE GROUP

h3_erbi = (
    h3_df
    .assign(
        erbi=lambda x:
            x["ed_visits"] *
            x["median_los_minutes"]
    )
    .groupby(
        "age_group",
        observed=True
    )["erbi"]
    .sum()
    .reindex(ordered_age_groups)
)

plt.figure(figsize=(12, 6))

bars = plt.bar(
    chart_labels,
    h3_erbi.values
)

plt.title(
    "Estimated Resource Burden Index by Age Group"
)

plt.xlabel(
    "Age Group"
)

plt.ylabel(
    "Estimated Resource Burden Index"
)

plt.xticks(rotation=35)

for bar, value in zip(
    bars,
    h3_erbi.values
):

    label = (
        f"{value / 1e9:.2f}B"
        if value >= 1e9
        else f"{value / 1e6:.2f}M"
    )

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        label,
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.show()


In [ ]:
# ED VISIT SHARE BY AGE GROUP

visit_share = (
    visit_summary /
    visit_summary.sum()
) * 100

plt.figure(figsize=(10, 8))

plt.pie(
    visit_share.values,
    labels=chart_labels,
    autopct="%1.1f%%",
    startangle=90
)

plt.title(
    "Share of Reported ED Visits by Age Group"
)

plt.tight_layout()
plt.show()


In [ ]:
# ED VISIT VOLUME VS REPORTED MEDIAN LOS BY AGE GROUP

scatter_data = pd.DataFrame({

    "Age_Group": chart_labels,

    "ED_Visits":
        visit_summary.values,

    "Weighted_Median_LOS":
        weighted_los.values

})

plt.figure(figsize=(11, 7))

plt.scatter(
    scatter_data["ED_Visits"],
    scatter_data["Weighted_Median_LOS"],
    s=120
)

for _, row in scatter_data.iterrows():

    plt.annotate(
        row["Age_Group"],
        (
            row["ED_Visits"],
            row["Weighted_Median_LOS"]
        ),
        xytext=(8, 8),
        textcoords="offset points"
    )

plt.xlabel(
    "Total Reported ED Visits"
)

plt.ylabel(
    "Weighted Median LOS (Minutes)"
)

plt.title(
    "ED Visit Volume vs Reported Median LOS by Age Group"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# H3 STATISTICAL RESULT

p_value = h3_result["p_value"]

epsilon_squared = (
    (h3_result["H"] - h3_result["df"])
    /
    (
        h3_result["N_weight"]
        - h3_result["df"]
        - 1
    )
)

decision = (
    "Reject H₀"
    if p_value < 0.05
    else
    "Fail to Reject H₀"
)

plt.figure(figsize=(10, 5))

plt.axis("off")

plt.text(
    0.05,
    0.75,
    "H3 — Age Groups vs Reported Median ED LOS",
    fontsize=16,
    fontweight="bold"
)

plt.text(
    0.05,
    0.58,
    f"Kruskal–Wallis H = {h3_result['H']:.4f}",
    fontsize=13
)

plt.text(
    0.05,
    0.46,
    f"Degrees of Freedom = {h3_result['df']}",
    fontsize=13
)

plt.text(
    0.05,
    0.34,
    f"P-value = {p_value:.6g}",
    fontsize=13
)

plt.text(
    0.05,
    0.22,
    f"ε² = {epsilon_squared:.6f}",
    fontsize=13
)

plt.text(
    0.05,
    0.10,
    f"Decision = {decision}",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()


## Summary

This notebook used the aggregate `Age_Sex` dataset to assess whether **reported median ED length of stay differs across age groups**.

- The dataset was loaded from the cleaned workbook or `Age_Sex.csv`.
- Aggregate observations were validated for missing and invalid values.
- ED visit counts were used as frequency weights for the descriptive summaries and weighted Kruskal-Wallis test.
- The analysis does **not** make patient-level claims.
- The formal test is a weighted Kruskal-Wallis H test across age-group categories.
- The conclusion depends on whether the p-value is below the chosen significance level of 0.05.

### Interpretation boundary

A statistically significant result would indicate that the **reported aggregate median ED LOS values differ across the age-group categories represented in the dataset**. It should not be interpreted as evidence that individual patients in one age group personally stay longer than patients in another age group.
